# V6.2E — BBox/Data Quality Audit Before RT-DETR

This notebook does **not train another model**.

It audits the exact accepted V6.2E dataset after the YOLO11 baselines showed a large localization gap.

Observed validation baselines:

| model | mAP50 | mAP50:95 | gap | retention |
|---|---:|---:|---:|---:|
| YOLO11n @640 | 0.768 | 0.413 | 0.355 | 53.8% |
| YOLO11m @1024 | 0.815 | 0.457 | 0.358 | 56.1% |

The goal is to determine whether the bottleneck is likely caused by:

- inconsistent bbox tightness between sources;
- very small defects;
- extreme aspect-ratio objects;
- duplicated / near-duplicated boxes;
- broken-vs-flashover annotation conflicts;
- large numbers of background-only images;
- class/source imbalance;
- source-specific annotation style.

This audit keeps TEST untouched for model evaluation. It only inspects dataset labels and image geometry.


In [ ]:
from pathlib import Path
import os, sys, json, csv, shutil, subprocess, hashlib, zipfile, math
from collections import Counter, defaultdict

W=Path("/kaggle/working")
TMP=Path("/tmp/v6p2e_bbox_audit")
BASE_EXTRACT=TMP/"base"
OVERLAY_EXTRACT=TMP/"overlay"
RECON_LABELS=TMP/"reconciled_labels"
OUT=W/"v6p2e_bbox_audit"

for p in [TMP,BASE_EXTRACT,OVERLAY_EXTRACT,RECON_LABELS,OUT]:
    p.mkdir(parents=True,exist_ok=True)

FINAL_TAG="v6p2e_23562a45b343"
PARENT_TAG="v6p1_5d0b1149891c"
BASE_TAG="56b7ea194ee5"
OVERLAY_TAG="ee2d3c4497fd"

DATA_ROOT="stash://bunpmc/projects/yolov11sdi/datasets"
FINAL_ROOT=f"{DATA_ROOT}/v6p2e/{FINAL_TAG}"
PARENT_ROOT=f"{DATA_ROOT}/v6p1/{PARENT_TAG}"

BASE_STASH=f"{DATA_ROOT}/curated_v6_rs1280_{BASE_TAG}.tar"
OVERLAY_STASH=f"{PARENT_ROOT}/v6p1_fotl_overlay_{OVERLAY_TAG}.tar.gz"
FREEZE_STASH=f"{FINAL_ROOT}/v6p2e.freeze.json"
MANIFEST_STASH=f"{FINAL_ROOT}/v6p2e_split_manifest.csv"
RECON_STASH=f"{FINAL_ROOT}/v6p2e_reconciled_labels.zip"

NAMES=[
    "insulator",
    "insulator_broken",
    "insulator_flashover",
    "bird_nest",
    "foreign_object",
    "broken_strand",
]

# Diagnostic thresholds only. These are not automatic scientific truth.
ULTRA_TINY_AREA=0.0001   # <0.01% image area
TINY_AREA=0.0005         # <0.05% image area
VERY_LARGE_AREA=0.50
EXTREME_AR=10.0
DUP_IOU=0.90
DEFECT_CONFLICT_IOU=0.50


## 1. Install dependencies + Camber

In [ ]:
subprocess.check_call([
    sys.executable,"-m","pip","install","-q",
    "pandas","pyyaml","pillow","matplotlib"
])

import pandas as pd
import numpy as np
import yaml
from PIL import Image, ImageDraw, ImageOps
import matplotlib.pyplot as plt

from kaggle_secrets import UserSecretsClient
key=UserSecretsClient().get_secret("CAMBER_API_KEY")
if not key:
    raise RuntimeError("Missing CAMBER_API_KEY")
os.environ["CAMBER_API_KEY"]=key

if shutil.which("camber") is None and not (Path.home()/".camber/bin/camber").exists():
    subprocess.check_call(
        "curl -sL https://cli.cambercloud.com/install-v2.sh | bash",
        shell=True
    )
os.environ["PATH"]=f"{Path.home()}/.camber/bin:{Path.home()}/.local/bin:"+os.environ.get("PATH","")
CAMBER=shutil.which("camber") or str(Path.home()/".camber/bin/camber")

def stash_cp(src,dst,required=True):
    dst=Path(dst); dst.parent.mkdir(parents=True,exist_ok=True)
    r=subprocess.run([CAMBER,"stash","cp",str(src),str(dst)],
                     text=True,capture_output=True,env=os.environ.copy())
    ok=r.returncode==0 and dst.exists()
    if required and not ok:
        raise RuntimeError(r.stderr[-1800:] or r.stdout[-1800:])
    return ok

def stash_put(src,dst):
    r=subprocess.run([CAMBER,"stash","cp",str(src),str(dst)],
                     text=True,capture_output=True,env=os.environ.copy())
    if r.returncode!=0:
        raise RuntimeError(r.stderr[-1800:] or r.stdout[-1800:])
    print("Uploaded:",dst)


## 2. Restore exact V6.2E lineage

In [ ]:
freeze_local=TMP/"v6p2e.freeze.json"
manifest_local=TMP/"v6p2e_split_manifest.csv"
recon_zip=TMP/"v6p2e_reconciled_labels.zip"
base_tar=TMP/f"base_{BASE_TAG}.tar"
overlay_tar=TMP/f"overlay_{OVERLAY_TAG}.tar.gz"

stash_cp(FREEZE_STASH,freeze_local)
stash_cp(MANIFEST_STASH,manifest_local)
stash_cp(RECON_STASH,recon_zip)

freeze=json.loads(freeze_local.read_text())
manifest=pd.read_csv(manifest_local)

assert freeze["final_tag"]==FINAL_TAG
assert freeze["freeze_ready"] is True

if not any(BASE_EXTRACT.rglob("data.yaml")):
    stash_cp(BASE_STASH,base_tar)
    subprocess.check_call(["tar","-xf",str(base_tar),"-C",str(BASE_EXTRACT)])
    base_tar.unlink(missing_ok=True)

if not any(OVERLAY_EXTRACT.rglob("data.yaml")):
    stash_cp(OVERLAY_STASH,overlay_tar)
    subprocess.check_call(["tar","-xzf",str(overlay_tar),"-C",str(OVERLAY_EXTRACT)])
    overlay_tar.unlink(missing_ok=True)

if RECON_LABELS.exists():
    shutil.rmtree(RECON_LABELS)
RECON_LABELS.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(recon_zip) as z:
    z.extractall(RECON_LABELS)

def norm_names(names):
    if isinstance(names,dict):
        return [names[k] for k in sorted(names,key=lambda x:int(x))]
    return list(names)

def find_root(root):
    for yp in Path(root).rglob("data.yaml"):
        try:
            cfg=yaml.safe_load(yp.read_text())
            if norm_names(cfg.get("names",[]))==NAMES:
                return yp.parent
        except Exception:
            pass
    raise RuntimeError(f"Six-class root not found under {root}")

BASE=find_root(BASE_EXTRACT)
OVERLAY=find_root(OVERLAY_EXTRACT)

print("BASE:",BASE)
print("OVERLAY:",OVERLAY)
print("Manifest rows:",len(manifest))


## 3. Reconstruct image/label paths

In [ ]:
def locate_original(row):
    old_path=str(row["image_path_source"]).replace("\\","/")
    old_split=str(row["previous_v6p1_split"])
    marker=f"/images/{old_split}/"
    if marker not in old_path:
        raise RuntimeError(old_path)

    suffix=old_path.split(marker,1)[1]
    if suffix.startswith("base/"):
        rel=suffix[len("base/"):]
        root=BASE
    elif suffix.startswith("fotl/"):
        rel=suffix[len("fotl/"):]
        root=OVERLAY
    else:
        raise RuntimeError(suffix)

    ip=root/"images"/old_split/rel
    lp=root/"labels"/old_split/Path(rel).with_suffix(".txt")
    return ip,lp

def locate_label(row,lp):
    rp=RECON_LABELS/f"{row['sha256']}.txt"
    if str(row.get("resolution","")).startswith("merged"):
        return rp
    return lp

records=[]
for idx,row in manifest.iterrows():
    ip,lp0=locate_original(row)
    lp=locate_label(row,lp0)
    if not ip.exists():
        raise FileNotFoundError(ip)
    if not lp.exists():
        raise FileNotFoundError(lp)

    records.append({
        "row_id":idx,
        "split":str(row["split"]),
        "source":str(row["source"]),
        "sha256":str(row["sha256"]),
        "image_path":str(ip),
        "label_path":str(lp),
        "resolution":str(row.get("resolution","")),
    })

rec_df=pd.DataFrame(records)
display(rec_df.groupby(["split","source"]).size().reset_index(name="images"))


## 4. Parse every bbox and derive geometry

The audit records normalized bbox area, aspect ratio, center, edge proximity and source.


In [ ]:
def parse_label(path):
    out=[]
    for ln,line in enumerate(Path(path).read_text(errors="ignore").splitlines(),1):
        if not line.strip(): continue
        p=line.split()
        if len(p)<5: continue
        cid=int(float(p[0]))
        x,y,w,h=map(float,p[1:5])
        out.append((cid,x,y,w,h,ln))
    return out

box_rows=[]
bg_rows=[]

for i,r in rec_df.iterrows():
    boxes=parse_label(r["label_path"])
    if not boxes:
        bg_rows.append(r.to_dict())
        continue

    with Image.open(r["image_path"]) as im:
        Wd,Hd=im.size

    for cid,x,y,w,h,ln in boxes:
        area=w*h
        ar=max(w/h,h/w) if w>0 and h>0 else float("inf")
        x1=x-w/2; y1=y-h/2; x2=x+w/2; y2=y+h/2
        edge=min(x1,y1,1-x2,1-y2)

        box_rows.append({
            **r.to_dict(),
            "class_id":cid,
            "class":NAMES[cid],
            "line_no":ln,
            "cx":x,"cy":y,"w":w,"h":h,
            "area":area,
            "aspect_ratio_extreme":ar,
            "edge_margin":edge,
            "pixel_w":w*Wd,
            "pixel_h":h*Hd,
            "pixel_area":area*Wd*Hd,
            "ultra_tiny":area<ULTRA_TINY_AREA,
            "tiny":area<TINY_AREA,
            "very_large":area>VERY_LARGE_AREA,
            "extreme_ar":ar>EXTREME_AR,
            "touches_or_crosses_edge":edge<=0.002,
        })

boxes_df=pd.DataFrame(box_rows)
bg_df=pd.DataFrame(bg_rows)

boxes_df.to_csv(OUT/"all_boxes_geometry.csv",index=False)
bg_df.to_csv(OUT/"background_images.csv",index=False)

print("Boxes:",len(boxes_df))
print("Background images:",len(bg_df))


## 5. Per-class geometry summary

In [ ]:
def q(s,p):
    return float(s.quantile(p)) if len(s) else float("nan")

rows=[]
for cls,g in boxes_df.groupby("class"):
    rows.append({
        "class":cls,
        "boxes":len(g),
        "area_q01":q(g.area,.01),
        "area_q10":q(g.area,.10),
        "area_median":q(g.area,.50),
        "area_q90":q(g.area,.90),
        "pixel_w_median":q(g.pixel_w,.50),
        "pixel_h_median":q(g.pixel_h,.50),
        "tiny_pct":100*g.tiny.mean(),
        "ultra_tiny_pct":100*g.ultra_tiny.mean(),
        "extreme_ar_pct":100*g.extreme_ar.mean(),
        "edge_touch_pct":100*g.touches_or_crosses_edge.mean(),
    })

class_geom=pd.DataFrame(rows).sort_values("area_median")
display(class_geom)
class_geom.to_csv(OUT/"class_geometry_summary.csv",index=False)


## 6. Source × class annotation-style summary

Large differences in median normalized box area for the same class can indicate source-specific annotation conventions.


In [ ]:
rows=[]
for (source,cls),g in boxes_df.groupby(["source","class"]):
    rows.append({
        "source":source,
        "class":cls,
        "boxes":len(g),
        "area_median":float(g.area.median()),
        "area_q10":float(g.area.quantile(.10)),
        "area_q90":float(g.area.quantile(.90)),
        "pixel_w_median":float(g.pixel_w.median()),
        "pixel_h_median":float(g.pixel_h.median()),
        "tiny_pct":100*g.tiny.mean(),
        "extreme_ar_pct":100*g.extreme_ar.mean(),
    })

source_class=pd.DataFrame(rows)
display(source_class.sort_values(["class","area_median"]))
source_class.to_csv(OUT/"source_class_geometry.csv",index=False)

# Relative median-style flag.
global_med=boxes_df.groupby("class").area.median().to_dict()
style_flags=[]

for _,r in source_class.iterrows():
    gm=global_med[r["class"]]
    ratio=r["area_median"]/gm if gm>0 else np.nan
    if r["boxes"]>=20 and (ratio<0.33 or ratio>3.0):
        style_flags.append({
            **r.to_dict(),
            "median_area_ratio_to_class_global":ratio,
            "flag":"SOURCE_BOX_SCALE_STYLE_OUTLIER"
        })

style_df=pd.DataFrame(style_flags)
display(style_df)
style_df.to_csv(OUT/"source_annotation_style_flags.csv",index=False)


## 7. Duplicate / conflict boxes within each image

In [ ]:
def xywh_to_xyxy(b):
    x,y,w,h=b
    return x-w/2,y-h/2,x+w/2,y+h/2

def iou(a,b):
    ax1,ay1,ax2,ay2=xywh_to_xyxy(a)
    bx1,by1,bx2,by2=xywh_to_xyxy(b)
    ix1=max(ax1,bx1); iy1=max(ay1,by1)
    ix2=min(ax2,bx2); iy2=min(ay2,by2)
    iw=max(0,ix2-ix1); ih=max(0,iy2-iy1)
    inter=iw*ih
    aa=max(0,ax2-ax1)*max(0,ay2-ay1)
    bb=max(0,bx2-bx1)*max(0,by2-by1)
    den=aa+bb-inter
    return inter/den if den>0 else 0.0

suspicious=[]

for sha,g in boxes_df.groupby("sha256"):
    rr=list(g.to_dict("records"))
    for i in range(len(rr)):
        for j in range(i+1,len(rr)):
            a,b=rr[i],rr[j]
            ov=iou((a["cx"],a["cy"],a["w"],a["h"]),
                   (b["cx"],b["cy"],b["w"],b["h"]))
            reason=None
            if a["class_id"]==b["class_id"] and ov>=DUP_IOU:
                reason="SAME_CLASS_DUPLICATE_BOX"
            elif {a["class_id"],b["class_id"]}=={1,2} and ov>=DEFECT_CONFLICT_IOU:
                reason="BROKEN_FLASHOVER_OVERLAP"
            if reason:
                suspicious.append({
                    "sha256":sha,
                    "split":a["split"],
                    "source":a["source"],
                    "image_path":a["image_path"],
                    "class_a":a["class"],
                    "class_b":b["class"],
                    "iou":ov,
                    "reason":reason,
                })

overlap_df=pd.DataFrame(suspicious)
display(overlap_df.head(50))
overlap_df.to_csv(OUT/"within_image_overlap_flags.csv",index=False)

print("Suspicious overlaps:",len(overlap_df))


## 8. Background-only audit by source and split

In [ ]:
if len(bg_df):
    bg_summary=(
        bg_df.groupby(["split","source"])
        .size()
        .reset_index(name="background_images")
        .sort_values(["split","background_images"],ascending=[True,False])
    )
else:
    bg_summary=pd.DataFrame(columns=["split","source","background_images"])

display(bg_summary)
bg_summary.to_csv(OUT/"background_summary.csv",index=False)


## 9. Candidate review queues

These queues do **not** mean the labels are wrong. They are the highest-value samples for manual review.


In [ ]:
review = pd.concat([
    boxes_df[boxes_df.ultra_tiny].assign(review_reason="ULTRA_TINY_BOX"),
    boxes_df[boxes_df.extreme_ar].assign(review_reason="EXTREME_ASPECT_RATIO"),
    boxes_df[boxes_df.very_large].assign(review_reason="VERY_LARGE_BOX"),
    boxes_df[boxes_df.touches_or_crosses_edge].assign(review_reason="EDGE_TOUCHING_BOX"),
],ignore_index=True)

review=review.drop_duplicates(["sha256","line_no","review_reason"])
review.to_csv(OUT/"bbox_review_queue.csv",index=False)

print("Review candidates:",len(review))
display(
    review.groupby(["review_reason","class"])
    .size().reset_index(name="count")
    .sort_values("count",ascending=False)
)


## 10. Draw review contact sheets

In [ ]:
def draw_boxes(image_path, rows):
    with Image.open(image_path) as im:
        im=ImageOps.exif_transpose(im).convert("RGB")
        draw=ImageDraw.Draw(im)
        Wd,Hd=im.size
        for r in rows:
            x,y,w,h=r["cx"],r["cy"],r["w"],r["h"]
            x1=(x-w/2)*Wd; y1=(y-h/2)*Hd
            x2=(x+w/2)*Wd; y2=(y+h/2)*Hd
            draw.rectangle([x1,y1,x2,y2],width=max(2,int(min(Wd,Hd)/400)))
            draw.text((x1,max(0,y1-12)),r["class"])
        return im

def make_sheet(df,name,max_images=36,thumb=(320,240)):
    if df.empty:
        return None

    shas=list(dict.fromkeys(df["sha256"].tolist()))[:max_images]
    ims=[]
    for sha in shas:
        g=boxes_df[boxes_df.sha256==sha]
        if g.empty: continue
        im=draw_boxes(g.iloc[0].image_path,g.to_dict("records"))
        im.thumbnail(thumb)
        canvas=Image.new("RGB",thumb)
        canvas.paste(im,((thumb[0]-im.width)//2,(thumb[1]-im.height)//2))
        ims.append(canvas)

    if not ims: return None
    cols=6
    rows=math.ceil(len(ims)/cols)
    sheet=Image.new("RGB",(cols*thumb[0],rows*thumb[1]))
    for i,im in enumerate(ims):
        sheet.paste(im,((i%cols)*thumb[0],(i//cols)*thumb[1]))

    p=OUT/f"{name}.jpg"
    sheet.save(p,quality=90)
    print("Saved:",p)
    return p

make_sheet(review[review.review_reason=="ULTRA_TINY_BOX"],"review_ultra_tiny")
make_sheet(review[review.review_reason=="EXTREME_ASPECT_RATIO"],"review_extreme_aspect")
make_sheet(review[review.review_reason=="VERY_LARGE_BOX"],"review_very_large")


## 11. Final audit summary

In [ ]:
summary={
    "dataset_tag":FINAL_TAG,
    "images":len(rec_df),
    "boxes":len(boxes_df),
    "background_images":len(bg_df),
    "background_rate":len(bg_df)/len(rec_df),
    "ultra_tiny_boxes":int(boxes_df.ultra_tiny.sum()),
    "tiny_boxes":int(boxes_df.tiny.sum()),
    "extreme_aspect_boxes":int(boxes_df.extreme_ar.sum()),
    "very_large_boxes":int(boxes_df.very_large.sum()),
    "edge_touching_boxes":int(boxes_df.touches_or_crosses_edge.sum()),
    "within_image_overlap_flags":len(overlap_df),
    "source_annotation_style_flags":len(style_df),
}

print(json.dumps(summary,indent=2))
(OUT/"audit_summary.json").write_text(json.dumps(summary,indent=2))

report=[
    "# V6.2E bbox/data-quality audit",
    "",
    f"- Dataset: `{FINAL_TAG}`",
    f"- Images: {summary['images']}",
    f"- Boxes: {summary['boxes']}",
    f"- Background-only images: {summary['background_images']} ({summary['background_rate']:.2%})",
    f"- Ultra-tiny boxes: {summary['ultra_tiny_boxes']}",
    f"- Tiny boxes: {summary['tiny_boxes']}",
    f"- Extreme-aspect boxes: {summary['extreme_aspect_boxes']}",
    f"- Very-large boxes: {summary['very_large_boxes']}",
    f"- Edge-touching boxes: {summary['edge_touching_boxes']}",
    f"- Within-image overlap flags: {summary['within_image_overlap_flags']}",
    f"- Source annotation-style flags: {summary['source_annotation_style_flags']}",
    "",
    "This notebook does not automatically declare flagged labels incorrect.",
    "Use the review queues and source/class geometry tables to decide whether V6.3 relabeling is justified.",
]
(OUT/"AUDIT_REPORT.md").write_text("\n".join(report))
print((OUT/"AUDIT_REPORT.md").read_text())


## 12. Persist audit artifacts to Camber

In [ ]:
REMOTE=f"stash://bunpmc/projects/yolov11sdi/reports/v6p2e/{FINAL_TAG}/bbox_quality_audit"

audit_zip=W/f"{FINAL_TAG}_bbox_quality_audit.zip"
with zipfile.ZipFile(audit_zip,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p,p.relative_to(OUT.parent))

stash_put(audit_zip,f"{REMOTE}/{audit_zip.name}")
stash_put(OUT/"audit_summary.json",f"{REMOTE}/audit_summary.json")
stash_put(OUT/"AUDIT_REPORT.md",f"{REMOTE}/AUDIT_REPORT.md")

print("✅ Audit uploaded:",REMOTE)
